In [1]:
import os
import glob
import pandas as pd
import numpy as np
from xgboost import XGBClassifier as xgb
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import subprocess
from tqdm import tqdm

In [2]:
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
OUTPUT_DIR = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\spec_pitch training data"
SMILE_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
CONFIG_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\spec_pitch.conf"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# FIND ALL .wav/.txt PAIRS
wav_files = glob.glob(os.path.join(BASE_DIR, "*.wav"))
txt_files = [f.replace('.wav', '_label_audacity.txt') for f in wav_files]
valid_pairs = [(wav, txt) for wav, txt in zip(wav_files, txt_files) if os.path.exists(txt)]
print(f"Found {len(valid_pairs)} valid .wav/.txt pairs")

Found 100 valid .wav/.txt pairs


In [3]:
#PROCESS 80% FOR TRAINING (FIRST 80 FILES) 
train_pairs = valid_pairs[:80]
test_pairs = valid_pairs[80:100]  # Last 20 for validation
all_train_dfs = []

print("Processing 80 training files...")
for wav_file, txt_file in tqdm(train_pairs, desc="Train files"):
    base_name = os.path.basename(wav_file).replace('.wav', '')
    output_csv = os.path.join(OUTPUT_DIR, f"{base_name}_energy.csv")
    
    # 1. Extract features with openSMILE
    cmd = [SMILE_PATH, "-C", CONFIG_PATH, "-I", wav_file, "-O", output_csv]
    subprocess.run(cmd, check=True, capture_output=True)
    
    # 2. Parse single-column CSV
    df = pd.read_csv(output_csv, header=None)
    df_split = df[0].str.split(";", expand=True)
    df_split.columns = [
    "frameIndex",
    "frameTime",
    "spectralFlux_log",
    "spectralCentroid_log",
    "spectralMaxPos_log",
    "spectralMinPos_log",
    "pcm_RMSenergy",
    "pcm_LOGenergy","F0direction","directionScore"
]
    
    # 3. Add wheeze labels from .txt
    label_df = pd.read_csv(txt_file, sep="\t", header=None, names=["start", "end", "label"])
    wheeze_intervals = label_df[label_df["label"] == "Wheeze"][["start", "end"]].values.tolist()
    
    df_split["frameTime"] = pd.to_numeric(df_split["frameTime"], errors='coerce')
    def is_wheeze(frame_times, intervals):
        labels = pd.Series(0, index=frame_times.index)
        for start, end in intervals:
            mask = (frame_times >= start) & (frame_times <= end)
            labels[mask] = 1
        return labels
    df_split["Wheeze"] = is_wheeze(df_split["frameTime"], wheeze_intervals)
    df_split = df_split[df_split["frameTime"].notna()].reset_index(drop=True)
    
    # Save processed file
    df_split.to_csv(output_csv, index=False)
    
    # Add file identifier
    df_split["file_id"] = base_name
    all_train_dfs.append(df_split)
    print(f"{base_name}: {len(df_split)} frames, {df_split['Wheeze'].sum()} wheezes")

Processing 80 training files...


Train files:   0%|          | 0/80 [00:00<?, ?it/s]

Train files:   1%|▏         | 1/80 [00:00<00:10,  7.22it/s]

steth_20181001_11_01_50: 1497 frames, 667 wheezes


Train files:   2%|▎         | 2/80 [00:00<00:09,  8.29it/s]

...

trunc_2019-05-31-14-07-30-L5_13: 1497 frames, 1292 wheezes
trunc_2019-05-31-14-07-30-L5_4: 1497 frames, 933 wheezes


Train files:  99%|█████████▉| 79/80 [00:09<00:00,  6.48it/s]

trunc_2019-05-31-14-37-49-L1_13: 1497 frames, 1012 wheezes
trunc_2019-05-31-14-37-49-L1_14: 1497 frames, 801 wheezes


Train files: 100%|██████████| 80/80 [00:09<00:00,  8.25it/s]

trunc_2019-05-31-14-37-49-L2_13: 1497 frames, 999 wheezes


In [4]:
# COMBINE ALL TRAINING DATA 
train_df = pd.concat(all_train_dfs, ignore_index=True)
print(f"\n TRAINING SET: {train_df.shape[0]:,} total frames across {len(train_pairs)} files")
print("Wheeze distribution:\n", train_df['Wheeze'].value_counts())

# Cast to numeric
num_cols = [
    "frameTime",
    "spectralFlux_log", "spectralCentroid_log",
    "spectralMaxPos_log", "spectralMinPos_log",
    "pcm_RMSenergy", "pcm_LOGenergy","F0direction","directionScore"
]
for c in num_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")

# Drop rows with missing key features
train_df = train_df.dropna(
    subset=[
    "frameTime",
    "spectralFlux_log", "spectralCentroid_log",
    "spectralMaxPos_log", "spectralMinPos_log",
    "pcm_RMSenergy", "pcm_LOGenergy","F0direction","directionScore"
]).reset_index(drop=True)


features = [
    "pcm_LOGenergy",
    "pcm_RMSenergy",
    "spectralFlux_log",
    "spectralCentroid_log",
    "spectralMaxPos_log",
    "spectralMinPos_log","F0direction","directionScore"
]

X_train = train_df[features]
y_train = train_df["Wheeze"]



 TRAINING SET: 119,760 total frames across 80 files
Wheeze distribution:
 Wheeze
1    66734
0    53026
Name: count, dtype: int64


In [7]:
print("\n Training...")

model = xgb(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42
)
model.fit(X_train, y_train, verbose=10)

# pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
# model = rf(
#     n_estimators=200,
#     max_depth=6,
#     max_samples=0.8,
#     max_features=0.8,
#     class_weight={0:1, 1:pos_weight},
#     random_state=42,
# )
# model.fit(X_train, y_train)

print("\n🔍 Processing test files...")
test_dfs = []

# UPDATED expected columns for pitch_energy CSV
expected_cols = [
    "frameIndex", "frameTime",
    "spectralFlux_log", "spectralCentroid_log",
    "spectralMaxPos_log", "spectralMinPos_log",
    "pcm_RMSenergy", "pcm_LOGenergy","F0direction","directionScore"
]

def parse_smile_csv(raw_csv_path, txt_path):
    """Read single-column SMILE CSV, split into 8 cols, add Wheeze, save wide CSV."""
    raw = pd.read_csv(raw_csv_path, header=None)
    df_split = raw[0].str.split(";", expand=True)
    df_split.columns = expected_cols  # <-- changed

    label_df = pd.read_csv(
        txt_path, sep="\t", header=None,
        names=["start", "end", "label"]
    )
    wheeze_intervals = (
        label_df[label_df["label"] == "Wheeze"][["start", "end"]]
        .values
        .tolist()
    )

    df_split["frameTime"] = pd.to_numeric(df_split["frameTime"], errors="coerce")
    df_split["Wheeze"] = is_wheeze(df_split["frameTime"], wheeze_intervals)
    df_split = df_split[df_split["frameTime"].notna()].reset_index(drop=True)

    df_split.to_csv(raw_csv_path, index=False)
    return df_split


for wav_file, txt_file in tqdm(test_pairs, desc="Test files"):
    base_name = os.path.basename(wav_file).replace(".wav", "")
    output_csv = os.path.join(OUTPUT_DIR, f"{base_name}_energy.csv")

    if not os.path.exists(output_csv):
        cmd = [SMILE_PATH, "-C", CONFIG_PATH, "-I", wav_file, "-O", output_csv]
        subprocess.run(cmd, check=True, capture_output=True)
        df_test = parse_smile_csv(output_csv, txt_file)
    else:
        df_test = pd.read_csv(output_csv)
        # if columns are not the wide format, regenerate and parse
        if not set(expected_cols).issubset(df_test.columns):
            cmd = [SMILE_PATH, "-C", CONFIG_PATH, "-I", wav_file, "-O", output_csv]
            subprocess.run(cmd, check=True, capture_output=True)
            df_test = parse_smile_csv(output_csv, txt_file)

    # UPDATED numeric casting for new feature set
    for column in expected_cols:
        df_test[column] = pd.to_numeric(df_test[column], errors="coerce")
    # UPDATED dropna subset: only new features
    df_test = df_test.dropna(
        subset=[
            "frameTime",
            "spectralFlux_log", "spectralCentroid_log",
            "spectralMaxPos_log", "spectralMinPos_log",
            "pcm_RMSenergy", "pcm_LOGenergy","F0direction","directionScore"
        ]
    ).reset_index(drop=True)

    df_test["file_id"] = base_name
    test_dfs.append(df_test)

test_df = pd.concat(test_dfs, ignore_index=True)
X_test = test_df[features]
y_test = test_df["Wheeze"]


 Training...

🔍 Processing test files...


Test files: 100%|██████████| 20/20 [00:02<00:00,  8.75it/s]


In [8]:
y_pred = model.predict(X_test)
print("\n RESULTS ")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


 RESULTS 
              precision    recall  f1-score   support

           0       0.60      0.34      0.43     13504
           1       0.60      0.82      0.69     16436

    accuracy                           0.60     29940
   macro avg       0.60      0.58      0.56     29940
weighted avg       0.60      0.60      0.58     29940


Confusion Matrix:
[[ 4555  8949]
 [ 2989 13447]]


In [9]:
def create_sequences_from_existing(X, y, seq_length=50):
    """Fixed: Convert DataFrame to numpy first, then create sequences"""
    sequences, labels = [], []
    
    # Convert DataFrame to numpy arrays (FIXES AttributeError)
    X_np = X.values if hasattr(X, 'values') else np.array(X)
    y_np = y.values if hasattr(y, 'values') else np.array(y)
    
    n_frames, n_features = X_np.shape
    
    print(f"Data converted - frames: {n_frames}, features: {n_features}")
    
    # Create overlapping sequences from entire dataset
    for i in range(0, n_frames - seq_length + 1, 10):  # Stride=10
        seq = X_np[i:i+seq_length].flatten()  # Now works with numpy
        wheeze_ratio = y_np[i:i+seq_length].mean()
        label = 1 if wheeze_ratio > 0.3 else 0
        sequences.append(seq)
        labels.append(label)
    
    return np.array(sequences), np.array(labels)

# Create sequences using ONLY your existing variables
Xtrain_seq, ytrain_seq = create_sequences_from_existing(X_train, y_train)
Xtest_seq, ytest_seq = create_sequences_from_existing(X_test, y_test)

print(f"Sequence shapes - Train: {Xtrain_seq.shape}, Test: {Xtest_seq.shape}")
print(f"Wheeze ratio - Train: {ytrain_seq.mean():.3f}, Test: {ytest_seq.mean():.3f}")


Data converted - frames: 119760, features: 8
Data converted - frames: 29940, features: 8
Sequence shapes - Train: (11972, 400), Test: (2990, 400)
Wheeze ratio - Train: 0.661, Test: 0.648


In [10]:
def extract_cnn_features(X_seq, n_features_orig):
    """CNN features using ONLY numpy on your sequence data"""
    cnn_features = []
    seq_len = 50
    
    for seq in X_seq:
        # Reshape back to (seq_len, n_features)
        seq_2d = seq.reshape(seq_len, n_features_orig)
        
        # Convolution (moving averages per feature)
        conv1 = np.array([np.mean(seq_2d[i:i+5], axis=0) for i in range(seq_len-4)])
        conv2 = np.array([np.mean(seq_2d[i:i+3], axis=0) for i in range(seq_len-2)])
        
        # Pooling + stats
        pool1 = np.mean(conv1[::5], axis=0)
        pool2 = np.mean(conv2[::5], axis=0)
        stats = np.concatenate([seq_2d.mean(0), seq_2d.std(0), seq_2d.max(0)])
        
        features = np.concatenate([pool1, pool2, stats])
        cnn_features.append(features)
    
    return np.array(cnn_features)

# Extract features using ONLY your sequence data
n_features = X_train.shape[1]  # Use existing feature count
Xtrain_cnn = extract_cnn_features(Xtrain_seq, n_features)
Xtest_cnn = extract_cnn_features(Xtest_seq, n_features)

print(f"CNN features - Train: {Xtrain_cnn.shape}, Test: {Xtest_cnn.shape}")


CNN features - Train: (11972, 40), Test: (2990, 40)


In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Scale using ONLY your existing variables + sequences
scaler = StandardScaler()
Xtrain_hybrid = scaler.fit_transform(Xtrain_cnn)
Xtest_hybrid = scaler.transform(Xtest_cnn)

# Class weights from your existing ytrain
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
scale_pos_weight = class_weights[1] / class_weights[0]

# XGBoost with CNN features (enhanced version of your existing model)
model_cnn_xgb = xgb(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc'
)

model_cnn_xgb.fit(Xtrain_hybrid, ytrain_seq)

# Evaluate
ypred_cnn = model_cnn_xgb.predict(Xtest_hybrid)
print("CNN-XGBoost Results (using ONLY your variables):")
print(classification_report(ytest_seq, ypred_cnn))
print("Confusion Matrix:\n", confusion_matrix(ytest_seq, ypred_cnn))

CNN-XGBoost Results (using ONLY your variables):
              precision    recall  f1-score   support

           0       0.55      0.14      0.22      1052
           1       0.67      0.94      0.78      1938

    accuracy                           0.66      2990
   macro avg       0.61      0.54      0.50      2990
weighted avg       0.62      0.66      0.58      2990

Confusion Matrix:
 [[ 146  906]
 [ 121 1817]]
